# 03 | U-Net training, evaluation, inference and reprojection

Trains a small U-Net on the saved patches with a masked loss (pixels without a label are ignored), evaluates on held-out patches, runs windowed inference over each tile and reprojects the output for the web map.

Run on a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q rasterio

In [ ]:
import os, glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score, cohen_kappa_score

BASE = "/content/drive/MyDrive/forest_loss_project"
PATCH_DIR = os.path.join(BASE, "patches")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

patch_files = sorted(glob.glob(os.path.join(PATCH_DIR, "*.npy")))
train_files, test_files = train_test_split(patch_files, test_size=0.25, random_state=42)

## Dataset

NDVI and EVI are rescaled to roughly 0 to 1, since unscaled inputs made the network collapse to one class in early runs. The valid mask is built from the label before `nan_to_num`, otherwise NaN labels silently become 0 (non-loss) and corrupt training.

In [ ]:
def normalise(x):
    x = x.copy()
    x[4] = (x[4] + 1) / 2                 # NDVI: [-1, 1] -> [0, 1]
    x[5] = np.clip((x[5] + 1) / 2, 0, 1)  # EVI
    return x


class PatchDS(Dataset):
    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        d = np.load(self.files[i])
        valid = ~np.isnan(d[6]) & ~np.isnan(d[:6]).any(axis=0)
        d = np.nan_to_num(d, nan=0.0)
        x = normalise(d[:6])
        y = d[6]
        return (torch.tensor(x, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(valid, dtype=torch.float32))


train_dl = DataLoader(PatchDS(train_files), batch_size=8, shuffle=True)
test_dl = DataLoader(PatchDS(test_files), batch_size=8)

## Model

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=6):
        super().__init__()
        self.d1 = DoubleConv(in_ch, 32)
        self.d2 = DoubleConv(32, 64)
        self.d3 = DoubleConv(64, 128)
        self.bott = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.u3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.c3 = DoubleConv(256, 128)
        self.u2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.c2 = DoubleConv(128, 64)
        self.u1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.c1 = DoubleConv(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.d1(x)
        e2 = self.d2(self.pool(e1))
        e3 = self.d3(self.pool(e2))
        b = self.bott(self.pool(e3))
        x = self.c3(torch.cat([self.u3(b), e3], 1))
        x = self.c2(torch.cat([self.u2(x), e2], 1))
        x = self.c1(torch.cat([self.u1(x), e1], 1))
        return self.out(x).squeeze(1)


model = UNet().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

## Training with a masked loss

In [ ]:
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(2.7).to(device), reduction="none")


def masked_loss(logits, y, valid):
    l = bce(logits, y) * valid
    return l.sum() / valid.sum().clamp(min=1)


opt = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 15
history = []

for ep in range(EPOCHS):
    model.train()
    total = 0
    for x, y, v in train_dl:
        x, y, v = x.to(device), y.to(device), v.to(device)
        opt.zero_grad()
        loss = masked_loss(model(x), y, v)
        loss.backward()
        opt.step()
        total += loss.item()
    history.append(total / len(train_dl))
    print(f"epoch {ep + 1}/{EPOCHS}  loss {history[-1]:.4f}")

torch.save(model.state_dict(), os.path.join(BASE, "unet_weights.pth"))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history)
plt.xlabel("Epoch"); plt.ylabel("Training loss")
plt.show()

## Masked evaluation on held-out patches

In [ ]:
model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y, v in test_dl:
        p = (torch.sigmoid(model(x.to(device))) > 0.5).cpu()
        m = v.bool()
        ys.append(y[m].numpy())
        ps.append(p[m].numpy())
ys, ps = np.concatenate(ys), np.concatenate(ps)

print(confusion_matrix(ys, ps))
print("Precision:", round(precision_score(ys, ps), 3))
print("Recall:   ", round(recall_score(ys, ps), 3))
print("Kappa:    ", round(cohen_kappa_score(ys, ps), 3))

## Windowed inference over each full tile

Each tile is read and written one window at a time, so memory stays flat. Output is NaN wherever the input is invalid, and nodata is set to NaN so the web map does not show a fake no-data colour.

In [ ]:
import rasterio
from rasterio.windows import Window

PRED_DIR = os.path.join(BASE, "predictions")
os.makedirs(PRED_DIR, exist_ok=True)
tile_paths = sorted(glob.glob(os.path.join(BASE, "kinabatangan_stack_2025_v2*.tif")))
WIN = 256

model.eval()
for tp in tile_paths:
    out_path = os.path.join(PRED_DIR, "pred_" + os.path.basename(tp))
    with rasterio.open(tp) as src:
        profile = src.profile.copy()
        profile.update(count=1, dtype="float32", nodata=np.nan, compress="lzw")
        with rasterio.open(out_path, "w", **profile) as dst:
            for r in range(0, src.height, WIN):
                for c in range(0, src.width, WIN):
                    h = min(WIN, src.height - r)
                    w = min(WIN, src.width - c)
                    win = Window(c, r, w, h)
                    d = src.read(list(range(1, 7)), window=win).astype("float32")
                    invalid = np.isnan(d).any(axis=0)
                    x = normalise(np.nan_to_num(d, nan=0.0))
                    # pad to a multiple of 8 for the three pooling steps
                    ph, pw = (-h) % 8, (-w) % 8
                    x = np.pad(x, ((0, 0), (0, ph), (0, pw)))
                    xt = torch.tensor(x[None], dtype=torch.float32).to(device)
                    with torch.no_grad():
                        prob = torch.sigmoid(model(xt))[0].cpu().numpy()[:h, :w]
                    prob[invalid] = np.nan
                    dst.write(prob.astype("float32"), 1, window=win)
                    del xt
                    torch.cuda.empty_cache() if device == "cuda" else None
    print("wrote", out_path)

## Reproject to Web Mercator and average-downsample for the web map

`gdalwarp` is not available in Colab by default, so this uses `rasterio.warp`.

In [ ]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

WEB_DIR = os.path.join(BASE, "predictions_web")
os.makedirs(WEB_DIR, exist_ok=True)
DST_CRS = "EPSG:3857"

for pp in sorted(glob.glob(os.path.join(PRED_DIR, "pred_*.tif"))):
    out_path = os.path.join(WEB_DIR, "web_" + os.path.basename(pp))
    with rasterio.open(pp) as src:
        transform, width, height = calculate_default_transform(
            src.crs, DST_CRS, src.width, src.height, *src.bounds)
        profile = src.profile.copy()
        profile.update(crs=DST_CRS, transform=transform, width=width, height=height)
        with rasterio.open(out_path, "w", **profile) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=transform, dst_crs=DST_CRS,
                resampling=Resampling.average,
            )
    print("wrote", out_path)